# Bobby Carrot RL Training (T4 GPU)

Trains PPO / Rainbow DQN on levels 1-7, tests on levels 8-10.

**Runtime**: Go to `Runtime > Change runtime type > T4 GPU`

## 1. Setup

In [ ]:
# Clone your repo (change URL to your actual repo)
!git clone https://github.com/Charan20510/Mini_Project_Game.git
%cd Mini_Project_Game

In [ ]:
# Install dependencies
!pip install torch numpy pygame --quiet

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Set up paths
import sys
sys.path.insert(0, '.')
sys.path.insert(0, 'Game_Python')

# Colab runs headless - set dummy display for pygame
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

# Quick import test
from Bobby_Carrot.rl_models.config import PPOConfig, RainbowConfig, TrainingConfig, LevelConfig
from bobby_carrot.rl_env import BobbyCarrotEnv
print("All imports OK")

## 2. Configure Training

**Strategy**: Combined training on levels 1-7 with curriculum learning.
- Start with levels 1-3 (easy)
- Auto-promote to harder levels as success rate improves
- Test on unseen levels 8-10

In [ ]:
# ============================================================
#  TRAINING CONFIGURATION - Adjust these as needed
# ============================================================

ALGORITHM = "ppo"          # "ppo" or "rainbow"
USE_ICM = False            # Enable curiosity-driven exploration
TOTAL_TIMESTEPS = 500_000  # Increase for better results (1M-2M for thorough training)

# Level split
TRAIN_LEVELS = [("normal", i) for i in range(1, 8)]   # normal 1-7
TEST_LEVELS  = [("normal", i) for i in range(8, 11)]  # normal 8-10

# Curriculum: start with first 3 levels, add more as agent improves
CURRICULUM_START = 3
CURRICULUM_THRESHOLD = 0.6  # 60% success rate to unlock more levels

print(f"Algorithm: {ALGORITHM.upper()}" + (" + ICM" if USE_ICM else ""))
print(f"Train levels: {len(TRAIN_LEVELS)} (normal 1-7)")
print(f"Test levels:  {len(TEST_LEVELS)} (normal 8-10)")
print(f"Timesteps: {TOTAL_TIMESTEPS:,}")
print(f"Curriculum: start with {CURRICULUM_START} levels, promote at {CURRICULUM_THRESHOLD:.0%}")

## 3. Train PPO (Recommended)

In [ ]:
from Bobby_Carrot.rl_models.config import PPOConfig, TrainingConfig, LevelConfig, ICMConfig
from Bobby_Carrot.rl_models.ppo import train_ppo

level_config = LevelConfig(
    train_levels=TRAIN_LEVELS,
    test_levels=TEST_LEVELS,
)

train_config = TrainingConfig(
    total_timesteps=TOTAL_TIMESTEPS,
    seed=42,
    device="cuda",  # T4 GPU
    curriculum=True,
    curriculum_start_levels=CURRICULUM_START,
    curriculum_promotion_threshold=CURRICULUM_THRESHOLD,
    curriculum_add_levels=2,
    max_steps_per_episode=300,
    eval_interval=25_000,
    eval_episodes_per_level=10,
    log_interval=5_000,
    checkpoint_every=50_000,
)

ppo_config = PPOConfig(
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_ratio=0.2,
    entropy_coeff=0.01,
    value_coeff=0.5,
    rollout_length=512,     # Larger rollouts since GPU has headroom
    n_epochs=4,
    minibatch_size=128,     # Larger batches for GPU efficiency
)

icm_config = ICMConfig(enabled=USE_ICM)

# TRAIN!
agent = train_ppo(ppo_config, train_config, level_config, icm_config)

## 3b. Alternative: Train Rainbow DQN

Uncomment and run this cell INSTEAD of the PPO cell above if you want Rainbow.

In [ ]:
# from Bobby_Carrot.rl_models.config import RainbowConfig, TrainingConfig, LevelConfig, ICMConfig
# from Bobby_Carrot.rl_models.rainbow import train_rainbow
#
# level_config = LevelConfig(
#     train_levels=TRAIN_LEVELS,
#     test_levels=TEST_LEVELS,
# )
#
# train_config = TrainingConfig(
#     total_timesteps=1_000_000,  # Rainbow needs more steps
#     seed=42,
#     device="cuda",
#     curriculum=True,
#     curriculum_start_levels=CURRICULUM_START,
#     curriculum_promotion_threshold=CURRICULUM_THRESHOLD,
#     max_steps_per_episode=300,
#     eval_interval=50_000,
#     log_interval=5_000,
#     checkpoint_every=100_000,
# )
#
# rainbow_config = RainbowConfig(
#     lr=6.25e-5,
#     batch_size=64,          # Larger batch for GPU
#     buffer_size=200_000,
#     n_step=3,
#     atom_size=51,
#     target_update_freq=2000,
#     learning_starts=5000,
# )
#
# icm_config = ICMConfig(enabled=USE_ICM)
# agent = train_rainbow(rainbow_config, train_config, level_config, icm_config)

## 4. Evaluate on Test Levels (8-10)

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

# Choose the checkpoint to evaluate
if ALGORITHM == "ppo":
    ckpt_path = "checkpoints/ppo/ppo_final.pt"
else:
    ckpt_path = "checkpoints/rainbow/rainbow_final.pt"

results = evaluate_agent(
    algo=ALGORITHM,
    checkpoint_path=ckpt_path,
    levels=TEST_LEVELS,
    episodes_per_level=20,
    max_steps=300,
    device_str="cuda",
)

## 5. Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = f"logs/{ALGORITHM}/training_log.csv"
df = pd.read_csv(log_path)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'{ALGORITHM.upper()} Training on Normal Levels 1-7', fontsize=14, fontweight='bold')

# Reward curve
axes[0, 0].plot(df['timestep'], df['avg_reward'], color='#2196F3', linewidth=1.5)
axes[0, 0].set_title('Average Reward')
axes[0, 0].set_xlabel('Timesteps')
axes[0, 0].grid(alpha=0.3)

# Success rate
axes[0, 1].plot(df['timestep'], df['avg_success'].astype(float) * 100, color='#4CAF50', linewidth=1.5)
axes[0, 1].set_title('Success Rate (%)')
axes[0, 1].set_xlabel('Timesteps')
axes[0, 1].set_ylim(-5, 105)
axes[0, 1].grid(alpha=0.3)

# Active levels (curriculum)
axes[1, 0].plot(df['timestep'], df['active_levels'], color='#FF9800', linewidth=1.5)
axes[1, 0].set_title('Active Levels (Curriculum)')
axes[1, 0].set_xlabel('Timesteps')
axes[1, 0].set_ylim(0, 8)
axes[1, 0].grid(alpha=0.3)

# Loss
if 'policy_loss' in df.columns:  # PPO
    axes[1, 1].plot(df['timestep'], df['policy_loss'].astype(float), label='Policy', color='#F44336')
    axes[1, 1].plot(df['timestep'], df['entropy'].astype(float), label='Entropy', color='#9C27B0')
    axes[1, 1].legend()
else:  # Rainbow
    axes[1, 1].plot(df['timestep'], df['loss'].astype(float), color='#F44336')
axes[1, 1].set_title('Loss')
axes[1, 1].set_xlabel('Timesteps')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to training_curves.png")

## 6. Download Trained Model

In [ ]:
# Zip checkpoints for download
import shutil
shutil.make_archive('trained_model', 'zip', '.', 'checkpoints')

# Auto-download in Colab
try:
    from google.colab import files
    files.download('trained_model.zip')
    files.download('training_curves.png')
    print("Downloads started!")
except ImportError:
    print("Not running in Colab. Files saved locally.")

print(f"\nCheckpoint: {ckpt_path}")
print("To use locally:")
print(f"  .venv\\Scripts\\python.exe -m Bobby_Carrot.rl_models.evaluate --algo {ALGORITHM} --checkpoint {ckpt_path} --render")

## 7. Extended Training (Optional)

If results aren't good enough, try:
1. **More timesteps**: Change `TOTAL_TIMESTEPS` to 1M-2M
2. **Enable ICM**: Set `USE_ICM = True` (helps with sparse rewards)
3. **Try Rainbow**: Better sample efficiency for this puzzle game
4. **Lower curriculum threshold**: Set `CURRICULUM_THRESHOLD = 0.5` for faster level progression